# User Segmentation

Segment paid subscribers by geo (T1 / WW), funnel_version, upsell_version.

⚠️ **COST SAFETY**: 10 GB hard cap, 7-day default window.

In [ ]:
from google.cloud import bigquery
PROJECT = 'hopeful-list-429812-f3'
MAX_BYTES = 10 * 1024**3
client = bigquery.Client(project=PROJECT)
job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES, use_query_cache=True)

def run(sql):
    return client.query(sql, job_config=job_config).to_dataframe()

In [ ]:
df = run('''
SELECT
  CASE
    WHEN JSON_VALUE(event_metadata, '$.country_code') IN
         ('AE','AT','AU','BH','BN','CA','CZ','DE','DK','ES','FI','FR',
          'GB','HK','IE','IL','IT','JP','KR','NL','NO','PT','QA','SA',
          'SE','SG','SI','US','NZ')
    THEN 'T1' ELSE 'WW' END                                AS geo,
  JSON_VALUE(event_metadata, '$.funnel_version')           AS funnel_version,
  COUNT(DISTINCT user_id)                                  AS subscribers
FROM `events.funnel-raw-table`
WHERE event_name = 'pr_funnel_subscribe'
  AND country_code != 'KZ'
  AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY geo, funnel_version
ORDER BY subscribers DESC LIMIT 50
''')
df